<a href="https://colab.research.google.com/github/JoviWZhu/20206RAG/blob/RAG-Hybird/FineWeb_Edu_Hybrid_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets sentence-transformers faiss-cpu rank_bm25 huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.2 MB/s eta 0:00:00


In [3]:
import os
import random
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi
from huggingface_hub import InferenceClient

# ==========================================
# 1. INITIALIZATION & SETUP
# ==========================================
# Replace with your free Hugging Face Read Token

from google.colab import userdata
HF_TOKEN_DEV = userdata.get('HF_TOKEN_DEV')

client = InferenceClient(token=HF_TOKEN_DEV)


print("⚡ Step 1: Loading an educational pool from FineWeb-Edu...")
# We load a slightly larger sample (500 documents) so the quiz has diverse topics
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
#dataset_head = dataset.take(500)

# Pick a random starting point somewhere in the massive dataset stream
random_skip_offset = random.randint(0, 10000)

# Stream 100 rows starting from that random location
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
dataset_head = dataset.skip(random_skip_offset).take(200)

documents = [doc["text"] for doc in dataset_head]
print(f"✅ Loaded {len(documents)} core documents into your local repository.")


⚡ Step 1: Loading an educational pool from FineWeb-Edu...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

✅ Loaded 200 core documents into your local repository.


In [4]:
# ==========================================
# 2. BUILD THE DUAL-RETRIEVAL BACKEND
# ==========================================
print("\n⚡ Step 2a: Initializing Semantic Search Channel (Dense Vector)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = embedding_model.encode(documents, convert_to_numpy=True)

# Normalize for Cosine Similarity inside an Inner Product space
faiss.normalize_L2(document_embeddings)
dimension = document_embeddings.shape[1]
semantic_index = faiss.IndexFlatIP(dimension)
semantic_index.add(document_embeddings)

print("⚡ Step 2b: Initializing Keyword Search Channel (Sparse BM25)...")
# Tokenize documents into lowercased word arrays for exact token identification
tokenized_corpus = [doc.lower().split(" ") for doc in documents]
bm25_index = BM25Okapi(tokenized_corpus)

print("⚡ Step 2c: Loading BGE Reranker Model onto local CPU memory...")
# This reads query-document sequences and scores contextual overlap via cross-attention matrix
reranker_model = CrossEncoder("BAAI/bge-reranker-base")
print("✅ Hybrid indexing and reranking layers are online.")



⚡ Step 2a: Initializing Semantic Search Channel (Dense Vector)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✅ Hybrid indexing and reranking layers are online.


In [9]:
print(len(tokenized_corpus[0]))

377


In [27]:
# ==========================================
# 3. CORE HYBRID ROUTING & COMPILATION LOGIC
# ==========================================
def run_hybrid_rag_search(user_query, top_n_candidates=10, final_top_k=2):
    print(f"\n🔍 Processing Hybrid Search Engine Query: '{user_query}'")

    # --- PATHWAY A: DENSE SEMANTIC RETRIEVAL ---
    query_embedding = embedding_model.encode([user_query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    _, semantic_indices = semantic_index.search(query_embedding, top_n_candidates)
    semantic_results = [documents[idx] for idx in semantic_indices[0]]

    # --- PATHWAY B: SPARSE KEYWORD RETRIEVAL ---
    tokenized_query = user_query.lower().split(" ")# Change corpus=documents to documents=documents
    keyword_results = bm25_index.get_top_n(tokenized_query, documents=documents, n=top_n_candidates)


    # --- CANDIDATE POOL FUSION ---
    # Merge both pipelines into a set structure to eliminate duplicate row indexes
    candidate_pool = list(set(semantic_results + keyword_results))
    print(f"✅ Retrieved {len(candidate_pool)} total candidates from sparse & dense channels.")

    # --- STEP 3: CROSS-ATTENTION RERANKING ---
    print("⚡ Reranking candidates using Cross-Encoder attention scoring...")
    rerank_pairs = [[user_query, doc] for doc in candidate_pool]
    rerank_scores = reranker_model.predict(rerank_pairs)

    # Sort document entries in descending order based on their true alignment scores
    ranked_indices = np.argsort(rerank_scores)[::-1]
    final_contexts = [candidate_pool[idx] for idx in ranked_indices[:final_top_k]]
    print(f"🎯 Isolated the top {final_top_k} highest-density context frames.")

    # --- STEP 4: GROUNDED STUDY GUIDE GENERATION ---
    context_str = "\n---\n".join(final_contexts)

    print("--- TEST: WHAT AM I SENDING TO THE LLM? ---")
    print(context_str)
    print("-------------------------------------------")

    # --- Old Prompt ---
    #system_prompt = (
    #    "You are an elite academic curriculum designer. Your task is to process the retrieved textbook fragments "
    #    "and draft an executive Study Guide for a student. Break down key terms, extract core principles, and "
    #    "synthesize the facts structured beautifully with markdown headers. Ground everything strictly in the text."
    #)

    # --- Updated Prompt ---
    system_prompt = (
    "You are a strict academic grader. Your absolute priority is factual precision. "
    "Read the provided text context and answer the user's question using ONLY the explicit facts written in the text. "
    "CRITICAL RULE: If the text discusses a different topic, you MUST refuse to answer and say exactly: "
    "'I cannot find the answer in the provided documents.' "
    "Do not extrapolate, do not assume, and do not use outside knowledge."
    )


    user_prompt = f"Textbook Context Passages:\n{context_str}\n\nTarget Subject/Concept: {user_query}\n\nDraft Study Guide:"

    print("⚡ Step 4: Routing prompt packet to Hugging Face Serverless endpoint...")
    try:
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=800,
            temperature=0.2
        )

        print("\n📚 [ENTERPRISE TEXTBOOK STUDY GUIDE]:")

        # BULLETPROOF PARSING LAYER: Handles object variations across all serverless providers
        if hasattr(response, 'choices') and len(response.choices) > 0:
            choice = response.choices[0]
            if hasattr(choice, 'message'):
                print(choice.message.content)
            elif isinstance(choice, dict) and 'message' in choice:
                print(choice['message']['content'])
            else:
                print(choice)
        else:
            # Direct string/dictionary fallback layout
            if isinstance(response, dict) and 'choices' in response:
                print(response['choices'][0]['message']['content'])
            else:
                print(response)

    except Exception as e:
        print(f"❌ Error communicating with Hugging Face API: {e}")

In [28]:
run_hybrid_rag_search("What are the definitions and core characteristics of physical properties or engineering frameworks?", final_top_k=2)


🔍 Processing Hybrid Search Engine Query: 'What are the definitions and core characteristics of physical properties or engineering frameworks?'
✅ Retrieved 19 total candidates from sparse & dense channels.
⚡ Reranking candidates using Cross-Encoder attention scoring...
🎯 Isolated the top 2 highest-density context frames.
--- TEST: WHAT AM I SENDING TO THE LLM? ---
Individual differences |
Methods | Statistics | Clinical | Educational | Industrial | Professional items | World psychology |
- Main article: Alcoholic intoxication
Drunkenness is the state of being intoxicated by consumption of alcohol to a degree that mental and physical faculties are noticeably impaired. Common symptoms may include slurred speech, impaired balance, poor coordination, flushed face, reddened eyes, reduced inhibition, hiccuping, and uncharacteristic behavior. Drunkenness can result in temporary experience of a wide range of emotion, ranging from anger, sadness, and depression to euphoria, lightheartedness and

In [20]:
# ==========================================
# 3 & 4. FIXED RAG EXECUTION FUNCTION WITH ALL VARIABLES
# ==========================================
def run_rag_query(user_question, top_k=2):
    print(f"\n🔍 User Question: '{user_question}'")
    print(f"⚡ Step 3: Searching local database for relevant FineWeb-Edu facts...")

    # 3a. Vectorize user query
    question_embedding = embedding_model.encode([user_question], convert_to_numpy=True)

    # Normalize for Cosine Similarity metric
    faiss.normalize_L2(question_embedding)

    # 3b. Search local index
    distances, indices = semantic_index.search(question_embedding, top_k)

    # 3c. Extract text matching indices (flattening matrix output using indices[0])
    retrieved_contexts = [documents[idx] for idx in indices[0]]
    print("✅ Facts retrieved successfully.")

    # 3d. PACKAGE IT UP: Construct strict prompt instruction
    context_str = "\n---\n".join(retrieved_contexts)

    print("--- TEST: WHAT AM I SENDING TO THE LLM? ---")
    print(context_str)
    print("-------------------------------------------")

    # CRITICAL FIX: Defining system_prompt clearly before loading it below
    system_prompt = (
        "You are an academic expert assistant. Answer the user's question using ONLY the provided text context from FineWeb-Edu. "
        "If the answer cannot be confidently derived from the context, reply with 'I cannot find the answer in the provided documents.' "
        "Do not make up facts or use outside knowledge."
    )

    user_prompt = f"Context from FineWeb-Edu:\n{context_str}\n\nQuestion: {user_question}\nAnswer:"

    # ==========================================
    # 4. EXTERNAL API GENERATION STEP (HUGGING FACE)
    # ==========================================
    print("⚡ Step 4: Sending the packaged prompt bundle to Hugging Face Serverless API...")

    try:
        # Using the direct client.chat_completion endpoint
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=500,
            temperature=0.2
        )

        print("\n🤖 [Llama 3.1 RAG Response]:")

        # Completely bulletproof type checking framework for choices output
        if hasattr(response, 'choices') and len(response.choices) > 0:
            choice = response.choices[0]
            if hasattr(choice, 'message'):
                print(choice.message.content)
            elif isinstance(choice, dict) and 'message' in choice:
                print(choice['message']['content'])
            else:
                print(choice)
        else:
            print(response)

    except Exception as e:
        print(f"❌ Error communicating with Hugging Face API: {e}")

# ==========================================
# 5. RUN THE QUERY OVER RE-INITIALIZED BLOCKS
# ==========================================
run_rag_query("What are the definitions and core characteristics of physical properties or engineering frameworks?", top_k=2)



🔍 User Question: 'What are the definitions and core characteristics of physical properties or engineering frameworks?'
⚡ Step 3: Searching local database for relevant FineWeb-Edu facts...
✅ Facts retrieved successfully.
⚡ Step 4: Sending the packaged prompt bundle to Hugging Face Serverless API...

🤖 [Llama 3.1 RAG Response]:
I cannot find the answer in the provided documents.


Here is why your Hybrid Search Engine returned an answer while your Standard Search Engine said "Cannot find."

# Reason 1: The Keyword Search (BM25) Actually Found It!
This is the most common reason and it proves your Hybrid system is working.

When you ran run_rag_query (Standard RAG), it used only semantic vector distances. Because we only loaded 100 or 200 documents, the mathematical vector search might have prioritized documents that shared a general concept but lacked the exact words to answer your question.

However, when you ran run_hybrid_rag_search:


1.   The BM25 algorithm specifically hunted for exact matches of your query words.
2.   BM25 may have successfully pulled a document containing those exact keywords that the semantic vector search completely ranked lower or missed
3.   Because the keyword path added this correct document to the candidate pool, the BGE Reranker recognized its high quality, bumped it to the top, and fed it to the LLM.

In this scenario, the answer did come from your documents! The Hybrid engine succeeded where the standard engine failed.

# Reason 2: The Merged Candidate Pool Confused the LLM (Prompt Leak)
If you look closely at the document context printed during your Hybrid run and confirm the facts are still completely missing, then you have hit a limitation known as Prompt Confusion or Attention Leakage.

In your standard RAG, you passed only top_k=2 documents directly to the LLM. The context window was tiny, clean, and easy for the LLM to verify.

In your Hybrid RAG script, you configured the retrieval to gather a combined candidate pool (top_n_candidates=10 from both algorithms). Even after deduplication, you might be passing up to 15–20 heavily dense, multi-topic document fragments through the BGE Reranker and into the LLM context pool.

When an LLM is flooded with a massive wall of text that is tangentially related (sharing academic keywords but not answering the core question), the attention mechanism inside models like Llama 3.1 can experience a "distraction" breakdown. Instead of strictly adhering to the system prompt's refusal rules, the model triggers its internal pre-training knowledge weights because it sees familiar keywords scattered across the context blocks.

# How to Verify Which One Happened

You can prove exactly where the answer came from by running a quick diagnostic test inside your function. Add a print statement right before Step 4 to inspect the final text your system selected:python# Add this line right before your LLM api call in run_hybrid_rag_search:

```
print("--- TEST: WHAT AM I SENDING TO THE LLM? ---")
print(context_str)
print("-------------------------------------------")
```

Use code with caution.If you see the answer inside that printed block: Your Hybrid engine won! The BM25 algorithm fetched the exact textbook page the vector database missed.If you do not see the answer in that block: The LLM hallucinated using its own brain because the massive context pool distracted it.To fix an attention leak, you can lower your final_top_k parameter (e.g., set final_top_k=1) to force the Reranker to pass only the single absolute best matched fragment to the LLM.

While the retrieved text contains words like "definition" and "frame," it is explaining a kinematical concept in physics (Inertial Frames of Reference). It does not actually contain any details defining generalized physical properties (like density, conductivity, or mass) or engineering frameworks (like structural code criteria or system design frameworks).
This means your Hybrid RAG system experienced Reason 2 (Attention/Prompt Leakage).

## What happened under the hood:

   1. Keyword Overlap: Your query asked for "definitions and core characteristics of physical properties or engineering frameworks."
   2. The Retrieval Succeeded (Technically): The BM25 algorithm or your semantic index matched on the heavy concentration of matching academic words: "definition," "frame," "properties," and "physical/dynamical." Because of those exact keyword hits, this passage was retrieved as a top match.
   3. The LLM Hallucinated (The Leak): Because you gave the LLM a large paragraph filled with dense physics equations and terms, the model got "distracted." Instead of recognizing that the text was about relativity and Newton's laws [1] rather than general engineering frameworks, it used its own pre-trained brain to creatively bridge the gap and write a plausible-sounding study guide.

## How to Fix It Instantly
To force the LLM to be as strict as your Standard RAG and say "I cannot find the answer," you need to strengthen the grounding constraint inside your system_prompt.
Update the system_prompt inside your run_hybrid_rag_search function to this more aggressive version:

system_prompt = (
    "You are a strict academic grader. Your absolute priority is factual precision. "
    "Read the provided text context and answer the user's question using ONLY the explicit facts written in the text. "
    "CRITICAL RULE: If the text discusses a different topic (e.g., if it talks about physics 'inertial frames' "
    "but the user asked about 'engineering frameworks'), you MUST refuse to answer and say exactly: "
    "'I cannot find the answer in the provided documents.' "
    "Do not extrapolate, do not assume, and do not use outside knowledge."
)

By explicitly telling the model to watch out for topic mismatches (even if the keywords look similar), it will properly trigger the refusal mechanism when the retrieved context isn't a true structural match.

------------------------------
If you run your code again with this updated system prompt, does the Hybrid RAG engine correctly refuse to answer and print 'I cannot find the answer...'?

